# Ch 2. API와 무엇이 다른가

**API 한 줄 호출과 직접 forward가 무엇을 다르게 보여주는지 — 토큰화·로짓·샘플링·메모리를 손으로 확인한다.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part1/ch02_vs_api.ipynb)

In [ ]:
# !pip install -q transformers torch

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device 확인
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 최소 예제 — 토큰 단위로 들여다보기

같은 프롬프트로 토큰화 + 로짓을 직접 본다. 외부 API 키 없이 SmolLM2-135M만으로 30초.

**무엇이 보였나**:
- 다음 토큰 분포가 **평평하다** (top-1이 18%). "확정적이지 않다"는 시그널.
- 동의어급 후보가 5개 — 어느 쪽으로 가도 동화 시작으로 자연스럽다는 모델의 "의견".
- API 응답에선 이중 하나만 무작위로 골라진 결과만 본다.

In [ ]:
# peek_inside.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F

name = "HuggingFaceTB/SmolLM2-135M"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

prompt = "Once upon a time"
ids = tok(prompt, return_tensors="pt").input_ids

# (1) 토큰화 결과 — 무엇이 뭐로 쪼개졌나
print("Tokens:", [tok.decode([t]) for t in ids[0]])

# (2) 한 번의 forward로 다음 토큰 분포 얻기
with torch.no_grad():
    logits = model(ids).logits[0, -1]   # 시퀀스 마지막 위치의 vocab 차원 분포
probs = F.softmax(logits, dim=-1)

# (3) Top-5 후보
top5 = torch.topk(probs, 5)            # topk 한 줄로 후보 5개. API로는 못 얻는 정보.
print("\nTop-5 next token candidates:")
for p, i in zip(top5.values, top5.indices):
    print(f"  {tok.decode([i]):>15s}  {p.item():.4f}")

In [ ]:
# 한국어 프롬프트로도 로짓 확인
prompt_ko = "옛날 옛적에"
ids_ko = tok(prompt_ko, return_tensors="pt").input_ids
print("Korean tokens:", [tok.decode([t]) for t in ids_ko[0]])

with torch.no_grad():
    logits_ko = model(ids_ko).logits[0, -1]
probs_ko = F.softmax(logits_ko, dim=-1)
top5_ko = torch.topk(probs_ko, 5)
print("\nTop-5 (Korean prompt):")
for p, i in zip(top5_ko.values, top5_ko.indices):
    print(f"  {tok.decode([i]):>15s}  {p.item():.4f}")

print(f"\nTop-1 확률 영어 프롬프트: {torch.topk(F.softmax(logits, dim=-1), 1).values[0].item():.4f}")
print(f"Top-1 확률 한국어 프롬프트: {torch.topk(probs_ko, 1).values[0].item():.4f}")

## 실전 — temperature · top-p를 직접 짜보기

API의 `temperature` · `top_p`는 추상적 단어지만 식으로는 5줄이다. 직접 짜서 같은 입력에 같은 효과가 나는지 확인.

**실험**: 같은 prompt에 `temperature`만 0.3, 0.7, 1.2로 바꿔 3번씩 돌려보자.

**관찰**:
- **T=0.3** — 거의 매번 같은 문장. "Once upon a time, there was a little girl..." 패턴 반복.
- **T=0.7** — 표준 동화 톤이지만 매번 다름. API의 default가 보통 이 부근.
- **T=1.2** — 가끔 비문·갑작스러운 화제 전환. 창의성 vs 안정성 트레이드오프가 손에 잡힌다.

In [ ]:
# sampling_from_scratch.py
import torch, torch.nn.functional as F

def sample(logits, temperature=1.0, top_p=1.0, top_k=0):
    """logits: (vocab,) 1D tensor. returns next token id."""
    # 1) Temperature
    logits = logits / max(temperature, 1e-5)

    # 2) Top-k
    if top_k > 0:
        kth = torch.topk(logits, top_k).values[-1]
        logits = torch.where(logits < kth, torch.full_like(logits, -1e10), logits)

    # 3) Top-p (nucleus)
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    cutoff = cumsum > top_p
    cutoff[1:] = cutoff[:-1].clone(); cutoff[0] = False  # 첫 토큰은 항상 살림
    sorted_probs[cutoff] = 0
    probs = torch.zeros_like(probs).scatter_(0, sorted_idx, sorted_probs)
    probs = probs / probs.sum()

    return torch.multinomial(probs, 1).item()

print("sample() function defined.")

In [ ]:
# temperature 실험 — 0.3 / 0.7 / 1.2
prompt = "Once upon a time"

for T in [0.3, 0.7, 1.2]:
    print(f"\n--- T={T} ---")
    for _ in range(3):
        ids = tok(prompt, return_tensors="pt").input_ids
        for _ in range(20):
            with torch.no_grad():
                logits = model(ids).logits[0, -1]
            nxt = sample(logits, temperature=T, top_p=0.9)
            ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
        print(tok.decode(ids[0]))

## 연습문제

1. 본인이 자주 쓰는 한국어 프롬프트 5개를 SmolLM2 토크나이저로 토큰화해 토큰 수를 측정하라. GPT-4 토크나이저(`tiktoken`)와도 비교. 차이가 가장 큰 케이스 한 개를 골라 **왜 그런가** 한 줄로 설명.
2. §4의 `peek_inside.py`를 한국어 프롬프트로 돌려보고, top-1 확률이 영어 프롬프트일 때와 어떻게 다른지 비교.
3. §5의 sampling 함수에 `repetition_penalty`를 추가하라 (이미 등장한 토큰의 logit을 1/penalty로 깎기). T=0.3일 때 반복이 줄어드는지 검증.
4. **(생각해볼 것)** 본인 회사의 한 작업을 골라 §7 결정 트리를 통과시켜라. "직접"으로 떨어졌으면 break-even 트래픽 산수도 계산.

In [ ]:
# 연습문제 1 — 한국어 프롬프트 토큰 수 비교
# !pip install -q tiktoken

prompts = [
    "안녕하세요, 오늘 날씨가 참 좋네요.",
    "이 상품은 환불이 가능한가요?",
    "인공지능 모델을 직접 훈련하는 방법을 알고 싶습니다.",
    "서울에서 부산까지 KTX로 얼마나 걸리나요?",
    "파이썬으로 데이터 전처리하는 코드를 짜줘.",
]

name = "HuggingFaceTB/SmolLM2-135M"
tok_smol = AutoTokenizer.from_pretrained(name)

print(f"{'Prompt':40s} | SmolLM2")
print("-" * 55)
for p in prompts:
    n_smol = len(tok_smol.encode(p))
    print(f"{p[:40]:40s} | {n_smol:4d}")

In [ ]:
# 연습문제 2 — 이미 위에서 확인 완료 (peek_inside 한국어 버전)
# 여기에 추가 분석 작성


In [ ]:
# 연습문제 3 — repetition_penalty 추가
def sample_with_rep_penalty(logits, generated_ids, temperature=1.0, top_p=1.0, repetition_penalty=1.0):
    """repetition_penalty: 이미 등장한 토큰의 logit을 1/penalty로 깎기"""
    if repetition_penalty != 1.0:
        for token_id in set(generated_ids):
            if logits[token_id] > 0:
                logits[token_id] /= repetition_penalty
            else:
                logits[token_id] *= repetition_penalty
    return sample(logits, temperature=temperature, top_p=top_p)

# T=0.3, rep_penalty 1.3으로 비교
prompt = "Once upon a time"
print("=== T=0.3, no penalty ===")
ids = tok.encode(prompt, return_tensors="pt")
for _ in range(30):
    with torch.no_grad():
        logits = model(ids).logits[0, -1].clone()
    nxt = sample(logits, temperature=0.3)
    ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
print(tok.decode(ids[0]))

print("\n=== T=0.3, rep_penalty=1.3 ===")
ids = tok.encode(prompt, return_tensors="pt")
generated = list(ids[0].numpy())
for _ in range(30):
    with torch.no_grad():
        logits = model(ids).logits[0, -1].clone()
    nxt = sample_with_rep_penalty(logits, generated, temperature=0.3, repetition_penalty=1.3)
    generated.append(nxt)
    ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
print(tok.decode(ids[0]))

In [ ]:
# 연습문제 4 — 결정 트리 + break-even 산수

def break_even_calls_per_day(gpu_monthly_cost_krw, api_cost_per_call_krw):
    """월 GPU 비용 / 호출당 API 비용 = 월 호출 수 break-even -> 일 단위로 환산"""
    monthly_calls = gpu_monthly_cost_krw / api_cost_per_call_krw
    daily_calls = monthly_calls / 30
    return monthly_calls, daily_calls

# 예시: GPU 월 100,000원, API 호출당 10원
gpu_cost = 100_000  # 원
api_per_call = 10   # 원
monthly, daily = break_even_calls_per_day(gpu_cost, api_per_call)
print(f"Break-even: 월 {monthly:,.0f}건 / 일 {daily:,.0f}건")
print("일 10,000건 이상이면 self-host가 경제적")